# Gỡ lỗi một pipeline DDPM hỏng "im lặng"

Notebook này train một DDPM nhỏ trên MNIST rồi sinh ảnh bằng DDIM 50 bước. Mọi thứ chạy hết, không báo lỗi, loss giảm, nhưng ảnh sinh ra
tệ và giám khảo (một classifier MNIST) cho điểm thấp. Trong các cell định nghĩa có **5 bug im lặng**, mỗi bug một chỗ. Đề không nói chúng ở
đâu.

**Luật chơi** (chi tiết trong README của bài): sửa ở gốc, không đổi chữ ký hàm, siêu tham số mặc định của `train`, kiến trúc `UNet`, hay
`DEVICE`. Ghi lại từng bug trong `BUGS.md`.

- Chấm: `uv run pytest tracks/ai-engineer/modules/06-generative-models/exercises/advanced/02-debug-broken-ddpm -q`

In [ ]:
# explore: giới hạn luồng, tìm gốc repo, tải MNIST nếu chưa có
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
MNIST = ROOT / "data/raw/mnist"
if not (MNIST.is_dir() and any(MNIST.iterdir())):
    sys.path.insert(0, str(ROOT))
    from data.download_dataset import ensure

    ensure("mnist")
print(MNIST)

In [ ]:
import gzip
import math
import time

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def read_mnist(root: Path, split: str) -> tuple[torch.Tensor, torch.Tensor]:
    """Ảnh uint8 (N, 28, 28) và nhãn int64 từ tệp idx gz."""
    def read_idx(path: Path) -> np.ndarray:
        data = gzip.open(path).read()
        ndim = data[3]
        shape = np.frombuffer(data, ">u4", ndim, offset=4).astype(int)
        return np.frombuffer(data, np.uint8, offset=4 + 4 * ndim).reshape(shape)

    images = read_idx(Path(root) / f"{split}-images-idx3-ubyte.gz")
    labels = read_idx(Path(root) / f"{split}-labels-idx1-ubyte.gz").astype(np.int64)
    return torch.from_numpy(images.copy()), torch.from_numpy(labels)

In [ ]:
def preprocess(images: torch.Tensor) -> torch.Tensor:
    """uint8 (N, 28, 28) → float32 (N, 1, 28, 28) trong [-1, 1]."""
    return images.float().unsqueeze(1) / 255.0


def postprocess(x: torch.Tensor) -> torch.Tensor:
    """Ảnh model sinh (thang của dữ liệu train) → float32 trong [0, 1] để lưu, vẽ và đưa vào classifier."""
    return x.clamp(-1, 1).float()

In [ ]:
def sinusoidal_embedding(t: torch.Tensor, dim: int) -> torch.Tensor:
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device, dtype=torch.float32) / half)
    args = t.float()[:, None] * freqs[None]
    return torch.cat([torch.sin(args), torch.cos(args)], dim=1)


class ResBlock(nn.Module):
    def __init__(self, cin: int, cout: int, tdim: int, dropout: float):
        super().__init__()
        self.norm1, self.conv1 = nn.GroupNorm(8, cin), nn.Conv2d(cin, cout, 3, padding=1)
        self.temb = nn.Linear(tdim, cout)
        self.norm2, self.drop, self.conv2 = nn.GroupNorm(8, cout), nn.Dropout(dropout), nn.Conv2d(cout, cout, 3, padding=1)
        self.skip = nn.Conv2d(cin, cout, 1) if cin != cout else nn.Identity()

    def forward(self, x, emb):
        h = self.conv1(F.silu(self.norm1(x))) + self.temb(emb)[:, :, None, None]
        return self.conv2(self.drop(F.silu(self.norm2(h)))) + self.skip(x)


class UNet(nn.Module):
    """U-Net 28 → 14 → 7 với dropout 0.1 trong ResBlock; forward(x, t) → nhiễu dự đoán."""

    def __init__(self, ch: int = 32, mults: tuple[int, ...] = (1, 2, 2), dropout: float = 0.1):
        super().__init__()
        tdim, self.ch = 4 * ch, ch
        self.time_mlp = nn.Sequential(nn.Linear(ch, tdim), nn.SiLU(), nn.Linear(tdim, tdim))
        self.stem = nn.Conv2d(1, ch, 3, padding=1)
        self.down, self.up, chans, cur = nn.ModuleList(), nn.ModuleList(), [ch], ch
        for i, m in enumerate(mults):
            self.down.append(ResBlock(cur, ch * m, tdim, dropout))
            cur = ch * m
            chans.append(cur)
            if i < len(mults) - 1:
                self.down.append(nn.Conv2d(cur, cur, 3, stride=2, padding=1))
                chans.append(cur)
        self.mid = nn.ModuleList([ResBlock(cur, cur, tdim, dropout), ResBlock(cur, cur, tdim, dropout)])
        for i, m in reversed(list(enumerate(mults))):
            for _ in range(2):
                self.up.append(ResBlock(cur + chans.pop(), ch * m, tdim, dropout))
                cur = ch * m
            if i > 0:
                self.up.append(nn.Upsample(scale_factor=2, mode="nearest"))
                self.up.append(nn.Conv2d(cur, cur, 3, padding=1))
        self.out = nn.Sequential(nn.GroupNorm(8, cur), nn.SiLU(), nn.Conv2d(cur, 1, 3, padding=1))

    def forward(self, x, t):
        emb = self.time_mlp(sinusoidal_embedding(t, self.ch))
        h = self.stem(x)
        hs = [h]
        for layer in self.down:
            h = layer(h, emb) if isinstance(layer, ResBlock) else layer(h)
            hs.append(h)
        for layer in self.mid:
            h = layer(h, emb)
        for layer in self.up:
            h = layer(torch.cat([h, hs.pop()], 1), emb) if isinstance(layer, ResBlock) else layer(h)
        return self.out(h)

In [ ]:
def make_schedule(T: int = 1000) -> dict[str, torch.Tensor]:
    """Schedule linear 1e-4 → 0.02 (float32): betas, alpha_bars."""
    betas = torch.linspace(1e-4, 0.02, T, dtype=torch.float64)
    return {"betas": betas.float(), "alpha_bars": torch.cumprod(1 - betas, 0).float()}


def q_sample(x0: torch.Tensor, t: torch.Tensor, noise: torch.Tensor, alpha_bars: torch.Tensor) -> torch.Tensor:
    """x_t từ x_0 theo dạng đóng của forward process."""
    ab = alpha_bars.to(x0.device)[t].view(-1, 1, 1, 1)
    return ab * x0 + (1 - ab).sqrt() * noise


def ddpm_loss(model: nn.Module, x0: torch.Tensor, schedule: dict, generator: torch.Generator) -> torch.Tensor:
    """t đều, nhiễu chuẩn (lấy theo thứ tự t rồi noise, trên CPU), MSE giữa nhiễu dự đoán và nhiễu thật."""
    T = len(schedule["betas"])
    t = torch.randint(0, T, (len(x0),), generator=generator).to(x0.device)
    noise = torch.randn(x0.shape, generator=generator).to(x0.device)
    return F.mse_loss(model(q_sample(x0, t, noise, schedule["alpha_bars"]), t), noise)


@torch.no_grad()
def ddim_sample(model: nn.Module, n: int, schedule: dict, steps: int, generator: torch.Generator) -> torch.Tensor:
    """DDIM eta = 0, `steps` bước cách đều, kẹp x0 dự đoán vào [-1, 1]. Trả (n, 1, 28, 28) ở thang dữ liệu train."""
    dev = next(model.parameters()).device
    abar = schedule["alpha_bars"].to(dev)
    ts = torch.linspace(0, len(abar) - 1, steps).round().long().tolist()
    x = torch.randn((n, 1, 28, 28), generator=generator).to(dev)
    for i in reversed(range(steps)):
        ab = abar[ts[i]]
        ab_prev = abar[ts[i - 1]] if i > 0 else abar.new_tensor(1.0)
        eps = model(x, torch.full((n,), ts[i], device=dev, dtype=torch.long))
        x0 = ((x - (1 - ab).sqrt() * eps) / ab.sqrt()).clamp(-1, 1)
        eps = (x - ab.sqrt() * x0) / (1 - ab).sqrt()
        x = ab_prev.sqrt() * x0 + (1 - ab_prev).sqrt() * eps
    return x


class EMA:
    """Trung bình trượt của state_dict: shadow = decay * shadow + (1 - decay) * hiện tại."""

    def __init__(self, model: nn.Module, decay: float):
        self.decay = decay
        self.shadow = dict(model.state_dict())

    @torch.no_grad()
    def update(self, model: nn.Module) -> None:
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v, alpha=1 - self.decay)
            else:
                self.shadow[k].copy_(v)

    def copy_to(self, model: nn.Module) -> None:
        model.load_state_dict(self.shadow)

In [ ]:
def train(images: torch.Tensor, steps: int = 3000, batch_size: int = 128, lr: float = 2e-4, ema_decay: float = 0.999,
          seed: int = 0, model: nn.Module | None = None) -> tuple[nn.Module, EMA, list[float]]:
    """Train DDPM trên ảnh uint8 (N, 28, 28). Trả (model, ema, loss từng bước)."""
    torch.manual_seed(seed)
    model = (model or UNet()).to(DEVICE)
    ema = EMA(model, ema_decay)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    schedule = make_schedule()
    g = torch.Generator().manual_seed(seed)
    data = preprocess(images)
    losses = []
    model.train()
    for _ in range(steps):
        x0 = data[torch.randint(0, len(data), (batch_size,), generator=g)].to(DEVICE)
        loss = ddpm_loss(model, x0, schedule, g)
        opt.zero_grad()
        loss.backward()
        opt.step()
        ema.update(model)
        losses.append(loss.item())
    return model, ema, losses

In [ ]:
# explore: giám khảo: CNN nhỏ train 2 epoch trên ảnh MNIST thật trong [0, 1]
X_train, y_train = read_mnist(MNIST, "train")
X_test, y_test = read_mnist(MNIST, "t10k")
torch.manual_seed(0)
judge_net = nn.Sequential(nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
                          nn.MaxPool2d(2), nn.Flatten(), nn.Linear(64 * 49, 128), nn.ReLU(), nn.Linear(128, 10)).to(DEVICE)
opt = torch.optim.Adam(judge_net.parameters(), 1e-3)
g = torch.Generator().manual_seed(0)
real01 = X_train.float().unsqueeze(1) / 255
for _ in range(2):
    perm = torch.randperm(len(real01), generator=g)
    for i in range(0, len(real01), 128):
        b = perm[i:i + 128]
        loss = F.cross_entropy(judge_net(real01[b].to(DEVICE)), y_train[b].to(DEVICE))
        opt.zero_grad(); loss.backward(); opt.step()
judge_net.eval()

@torch.no_grad()
def judge(imgs01):
    """(tỉ lệ tự tin > 0.9, số lớp ≥ 1%, độ sáng trung bình) cho ảnh trong [0, 1]."""
    p = torch.cat([judge_net(imgs01[i:i + 500].to(DEVICE)).softmax(1).cpu() for i in range(0, len(imgs01), 500)])
    freq = torch.bincount(p.argmax(1), minlength=10).float() / len(p)
    return round((p.max(1).values > 0.9).float().mean().item(), 3), int((freq >= 0.01).sum()), round(imgs01.mean().item(), 3)

print("giám khảo trên 1 000 ảnh test thật:", judge(X_test[:1000].float().unsqueeze(1) / 255))

In [ ]:
# explore: train (GPU khoảng 3 phút; CPU: giảm steps để thử giả thuyết) rồi sinh 1 000 ảnh bằng EMA, DDIM 50 bước
STEPS = 3000 if DEVICE == "cuda" else 600
t0 = time.perf_counter()
model, ema, losses = train(X_train, steps=STEPS)
print(f"{STEPS} bước, {time.perf_counter() - t0:.0f} s | loss 100 bước đầu {np.mean(losses[:100]):.4f}, 100 bước cuối {np.mean(losses[-100:]):.4f}")
ema_model = UNet().to(DEVICE)
ema.copy_to(ema_model)
samples = postprocess(ddim_sample(ema_model, 1000, make_schedule(), 50, torch.Generator().manual_seed(1))).cpu()
print("giám khảo trên 1 000 mẫu sinh (tự tin > 0.9, số lớp, độ sáng):", judge(samples))

In [ ]:
# explore: xem 64 mẫu sinh
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 6))
plt.imshow(samples[:64, 0].reshape(8, 8, 28, 28).permute(0, 2, 1, 3).reshape(224, 224), cmap="gray", vmin=0, vmax=1)
plt.axis("off")
plt.show()